# Direct TS Prediction

Instead of searching along a path, we generate TS candidates directly using an AI model.

## How does it work?

tsdiff uses a diffusion model (hence the name). Diffusion models are very well suited for this task, because multiple transition states are often close to each other in energy even though they sometimes belong to different reaction classes, e.g. substitution vs. elimination. The diffusion model will start from a random guess, which it prints out and is very far away from any sensible chemical structure, and in each iteration improves the structures through denoising it. Think about it as the network actively diffusing the input towards the output (again hence the name). On top of that tsdiff does a good job of identifying reaction-patterns, by mapping the molecule onto a 2D graph.

Now we are testing it for a proton transfer between hydronium and water as well as a proton transfer between CH4 and CH3.

If you have a GPU I once again highly recommend using it, by setting the pytorch device.

In [ ]:
# create train, validation and test set, but in this case
# we are only interested in testing custom cases, so only
# the test set will contain data.

from helpers.preprocessing_helper import preprocess_args, preprocess
from helpers.sampling_helper import sampling_args, sample

In [ ]:
# class attributes contain parameters
prep_params = preprocess_args()
prep_params.train = 0.0
prep_params.valid = 0.0
# generate preprocessed files
preprocess(prep_params)

In [ ]:
# run a pretrained model on the generated test data
sample_params = sampling_args()
#sample_params.device = "cuda"
# only use one model for faster calculation
# if you want to use all 8 available ones, just uncomment this line
sample_params.ckpt = [sample_params.ckpt[0]]
sample_params.n_steps = 5000
sample(sample_params)

In [ ]:
# visualize the generated transition state structures
import py3Dmol


def view_xyz(filename):
    with open(filename, "r") as f:
        xyz = f.read()

        viewer = py3Dmol.view(width=500, height=400)
        viewer.addModel(xyz, "xyz")

        # nice default style
        viewer.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.3}})

        viewer.zoomTo()
    return viewer


view_xyz("ts0.xyz")

In [ ]:
view_xyz("ts1.xyz")

## Clustering and Training

A nice feature of the diffusion methodology is that when running this multiple times, different transition state structures will be found, if the PES is reasonably complex. If that is the case you can use tsdiffs clustering tool, which groups the reactions e.g. into competing reaction channels. Our test reactions are, however, too simple for that.

Regarding training there is, to the best of my knowledge, no short retraining opportunity for tsdiff, so if you want to add something you have to fully retrain from your seed.

If you are interested you can find the GitHub repository for tsdiff [here](https://github.com/seonghann/tsdiff).

## Where Do TS Predictors Work Well?

| System Type | Performance |
|------------|------------|
| Small organic reactions | Excellent |
| Proton transfer | Very strong |
| SN2 reactions | Strong |
| Rearrangements | Mixed |
| Large flexible systems | Challenging |
| Transition metals | Poor |
| Charged/solvent systems | Difficult |

Conclusion:
AI TS predictors are powerful but require hybrid workflows and validation.

# Now try tsdiff for your own reaction

Now for the fun part, where you create your own reaction and can use tsdiff to predict the transition state structure.

Therefore you will have to do the following:
- Create SMARTS for the reactions you are interested in, e.g. with ChatGPT
- Create xyz structure of reactants/products/whatever (this is just for workflow consistency with tsdiff)
- Create your new "test set" with the preprocessing routine
- Generate the transition state structure using the sampling routine

You can just append smarts.csv and structures.xyz in data/raw_data/ with the new smarts and xyz, but **beware of the atom ordering** in the smarts and the xyz! The atoms in the smarts need to have the same enumeration for reactants and products, and this ordering also needs to apply for the xyz coordinates. To give an example, if the reactant side of the smart has an O at position 2 and H at position three, but it's vice versa on the product side, this won't work.

In order to (only) calculate your new test reaction, you also need to adjust end_idx from 2 to 3.

# Gaussian Process Regression for optimising transition states

Gaussion process regression (GPR) is a machine learning technique that can be used to interpolate the potential energy surface between different known reference points. A benefit to this technique is that every evaluation gives an uncertainty, so that when the uncertainty is too high, a new reference point can be evaluated. This way, GPR 'learns' the PES during the sampling and can speed up optimisations by skipping some expensive energy calculations. Crucially, it does not require any pre-trained models.

Autoforce is a project that implements GPR for the NEB method and hooks it up to ASE. Clone the following repo and pip install it in your environment: https://github.com/amirhajibabaei/AutoForce. ASE is a python interface for different QC and MD codes. 

    git clone https://github.com/amirhajibabaei/AutoForce
    cd autoforce
    pip install .

You can set up your own calculators, which ASE can then use to calculate energies and gradients. In the folder of this repository you can find `vlxcalculator.py` which implements the `VeloxChemCalculator` class which calculates energies and gradients through VeloxChem. Try it if you want to! There is also the `MolecularActiveCalculator`. This is a subclass of the `ActiveCalculator` from ASE which implements GPR, but adjusted to deal with non-periodic systems.

In [ ]:
from ase.optimize import FIRE, BFGS
from ase.mep.neb import idpp_interpolate

from helpers.vlxcalculator import VeloxChemCalculator
from helpers.molactivecalculator import MolecularActiveCalculator
# from xtb.ase.calculator import XTB
import veloxchem as vlx

In [ ]:
# reactant = ...
# product = ...
reactant.set_cell([10, 10, 10])
product.set_cell([10, 10, 10])
charge = -1
multiplicity = 1

# ---- ActiveCalculator wrapping VeloxChem ----
# species: atomic numbers for Br(35), C(6), H(1), Cl(17)
# SubSeSoapKernel is used per species when 'species' is given — ~10x faster

active_calc = MolecularActiveCalculator(
    calculator=XTB(
        method="GFN2-xTB",
        charge=charge,
        multiplicity=multiplicity,
    ),

    # Uncomment the following line to use the VeloxChemCalculator instead
    # calculator = VeloxChemCalculator(xcfun='b3lyp',basis='def2-svp',charge=charge, multiplicity=multiplicity,mute=False,)
    ediff=0.05,  # energy sensitivity for sampling LCEs, eV
    fdiff=0.05,  # force sensitivity for triggering VeloxChem calls, eV/Å
    noise_f=0.05,  # target MAE for force fitting, eV/Å
    logfile='active.log',
    # pckl='model.pckl',  # saves/loads model automatically between runs
    # tape='model.sgpr',  # lightweight text record of all training data
    stdout=True,  # print log to stdout as well as logfile
    ioptim=1,  # optimise hyperparameters when new data is sampled
)

# ---- Build NEB image chain ----
N_IMAGES = 8

images = [reactant.copy()]
for _ in range(N_IMAGES):
    image = reactant.copy()
    images.append(image)
images.append(product.copy())
for image in images:
    image.calc = active_calc

# ---- Interpolate initial path using IDPP ----
neb = NEB(images,
          method="improvedtangent",
          climb=False,
          allow_shared_calculator=True)
idpp_interpolate(neb)

# ---- Phase 1: converge the band without climbing image ----
opt = BFGS(neb, trajectory="neb_sgr.traj", logfile="neb_sgr.log", maxstep=0.02)

opt.run(fmax=0.10)

# ---- Phase 2: climbing image to sharpen the TS ----
neb.climb = True
opt = BFGS(neb,
           trajectory="neb_ci_sgr.traj",
           logfile="neb_ci_sgr.log",
           maxstep=0.01)
opt.run(fmax=0.05)

05/08/2026 10:41:51 0 active calculator says Hello!
05/08/2026 10:41:51 0 kernel: [SeSoapKernel(3, 3, 4, 6.0, a=None, radii=DefaultRadii(1.0, {1: 0.5}), normalize=True)]
05/08/2026 10:41:51 0 settings:  ediff: 0.05  ediff_tot: 0.172  fdiff: 0.05 
05/08/2026 10:41:51 0 model size: 0 0
05/08/2026 10:41:51 0 exact energy: -274.6429440503342
05/08/2026 10:41:52 0 seed size: 1 3 details: [(0, np.int64(8)), (1, np.int64(1)), (2, np.int64(1))]
05/08/2026 10:41:52 0 1 added 0 randomly displaced LCEs
05/08/2026 10:41:52 0 2 added 0 randomly displaced LCEs
05/08/2026 10:41:52 0 3 added 0 randomly displaced LCEs
05/08/2026 10:41:52 0 4 added 1 randomly displaced LCEs
05/08/2026 10:41:52 0 4 added 2 randomly displaced LCEs
05/08/2026 10:41:53 0 4 added 3 randomly displaced LCEs
05/08/2026 10:41:53 0 added 3 randomly displaced LCEs


/home/rocky/code/Retreat/Nebs/AutoForce/theforce/regression/gppotential.py:1278: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1762103814636/work/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  return float(loss), g.view(-1).numpy()


05/08/2026 10:41:53 0 kernel normalization status True
05/08/2026 10:41:53 0 -274.6429440503342 0.0 1.3138792865055356e-08 
05/08/2026 10:41:55 1 added indu: 5 (0,5) -> size: 1 11 details: 0.15 
05/08/2026 10:41:58 1 DF: 0.2121216519418496  accept: 1
05/08/2026 10:42:00 1 exact energy: -268.95119248907446
05/08/2026 10:42:00 1 errors (pre):  del-E: -7  max|del-F|: 8.1  mean|del-F|: 2.6
05/08/2026 10:42:00 1 added data: 1 -> size: 2 11
05/08/2026 10:42:00 1 fit error (mean,mae): E: -0.00045 0.013   F: -0.077 0.65   R2: 0.7262
05/08/2026 10:42:00 1 noise: {'all': 0.24184371268455085}
05/08/2026 10:42:00 1 mean: AutoMean({8: -28.285483179929248, 1: -56.570966359858495})
05/08/2026 10:42:00 1 -268.87811031679814 0.0 0.024255575869447685 
05/08/2026 10:42:02 2 added indu: 3 (0,3) -> size: 2 14 details: 0.053 
05/08/2026 10:42:06 2 DF: 0.02262068379076533  accept: 0
05/08/2026 10:42:07 2 fit error (mean,mae): E: -0.0088 0.13   F: -0.095 0.55   R2: 0.7977
05/08/2026 10:42:07 2 noise: {'all': 